# 01b Prepare RegGS Scene (DL3DV-2)

Goal:

1. adapt `DL3DV-2` into a RegGS-ready scene;
2. preserve the **full sequence** (306 frames);
3. prefer **symlink over copy** when images do not need pixel changes;
4. export clean `intrinsics.json` and `cameras.json`;
5. verify the exported scene layout.

This notebook mirrors `01_prepare_reggs_scene.ipynb` for Re10k-1, adapted for DL3DV-2's naming conventions.

Canonical prepared-data path:

```text
dataset/DL3DV-2/part2/<scene_name>/
```

Source format notes:
- images: `rgb/frame_00001.png` (1-indexed, 5-digit padding)
- intrinsics: separate `intrinsics.json` with normalized values
- cameras: list with `cam_id` starting from 1, `image_name` as `frame_00001.png`
- images are 256x256, matching RegGS expectations

In [3]:
from pathlib import Path
import json
import os
import shutil
import struct

CV_ROOT = Path('~/CV_Project').expanduser()
DATASET_ROOT = CV_ROOT / 'dataset'

SOURCE_ROOT = DATASET_ROOT / 'DL3DV-2'
SOURCE_IMAGE_DIR = SOURCE_ROOT / 'rgb'
SOURCE_INTRINSICS_JSON = SOURCE_ROOT / 'intrinsics.json'
SOURCE_CAMERAS_JSON = SOURCE_ROOT / 'cameras.json'
PART2_ROOT = SOURCE_ROOT / 'part2'

PART2_ROOT.mkdir(parents=True, exist_ok=True)

print('SOURCE_ROOT =', SOURCE_ROOT)
print('SOURCE_IMAGE_DIR =', SOURCE_IMAGE_DIR)
print('PART2_ROOT =', PART2_ROOT)

SOURCE_ROOT = /home/bzhang512/CV_Project/dataset/DL3DV-2
SOURCE_IMAGE_DIR = /home/bzhang512/CV_Project/dataset/DL3DV-2/rgb
PART2_ROOT = /home/bzhang512/CV_Project/dataset/DL3DV-2/part2


## 1. Export configuration


In [4]:
EXPORT_SCENE_NAME = 'reggs_dl3dv2_fullseq_256'
EXPORT_ROOT = PART2_ROOT / EXPORT_SCENE_NAME
EXPORT_IMAGE_DIR = EXPORT_ROOT / 'images'

# DL3DV-2 source images are already 256x256, so the default path can use symlinks.
# If you later change target size, set TARGET_SIZE to (W, H) and the notebook will write resized images instead.
TARGET_SIZE = None
OVERWRITE_EXPORT = True

print('EXPORT_ROOT =', EXPORT_ROOT)
print('TARGET_SIZE =', TARGET_SIZE)
print('OVERWRITE_EXPORT =', OVERWRITE_EXPORT)

EXPORT_ROOT = /home/bzhang512/CV_Project/dataset/DL3DV-2/part2/reggs_dl3dv2_fullseq_256
TARGET_SIZE = None
OVERWRITE_EXPORT = True


## 2. Helpers

In [5]:
def read_png_size(path: Path):
    png_sig = bytes.fromhex('89504E470D0A1A0A')
    with path.open('rb') as f:
        sig = f.read(8)
        if sig != png_sig:
            raise ValueError(f'Not a PNG file: {path}')
        _length = struct.unpack('>I', f.read(4))[0]
        chunk_type = f.read(4)
        if chunk_type != b'IHDR':
            raise ValueError(f'Invalid PNG header: {path}')
        width, height = struct.unpack('>II', f.read(8))
    return width, height


def safe_reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def symlink_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)


def load_dl3dv_source():
    """Load DL3DV-2 source data.
    
    DL3DV-2 uses 1-indexed naming (frame_00001.png) while we export as 0-indexed (0000.png).
    """
    intrinsics = json.loads(SOURCE_INTRINSICS_JSON.read_text(encoding='utf-8'))
    cameras = json.loads(SOURCE_CAMERAS_JSON.read_text(encoding='utf-8'))
    
    # Sort cameras by image_name to ensure correct order
    cameras = sorted(cameras, key=lambda x: x['image_name'])
    
    # Build frame paths from cameras (DL3DV uses frame_00001.png naming)
    frame_paths = [SOURCE_IMAGE_DIR / cam['image_name'] for cam in cameras]
    missing = [str(p) for p in frame_paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Missing source images: {missing[:5]}')

    width, height = read_png_size(frame_paths[0])

    return {
        'cameras': cameras,
        'frame_paths': frame_paths,
        'source_width': width,
        'source_height': height,
        'intrinsics': intrinsics,
    }


source = load_dl3dv_source()
print('num frames =', len(source['frame_paths']))
print('source size =', (source['source_width'], source['source_height']))
print('intrinsics =', source['intrinsics'])

num frames = 306
source size = (256, 256)
intrinsics = {'cx': 0.5, 'cy': 0.5, 'fx': 0.7970605911654538, 'fy': 0.7972243051546298}


## 3. Inspect source scene

In [6]:
print('first five source frames:')
for p in source['frame_paths'][:5]:
    print(' -', p.name)

print()
print('first camera entry:')
print(json.dumps(source['cameras'][0], indent=2))

first five source frames:
 - frame_00001.png
 - frame_00002.png
 - frame_00003.png
 - frame_00004.png
 - frame_00005.png

first camera entry:
{
  "cam_id": 1,
  "cam_quat": [
    0.021004781126976013,
    0.021099701523780823,
    0.01128463540226221,
    0.9994930028915405
  ],
  "cam_trans": [
    -4.861327171325684,
    -0.5403552055358887,
    -2.9286084175109863
  ],
  "cx": 0.5,
  "cy": 0.5,
  "fx": 0.7970605911654538,
  "fy": 0.7972243051546298,
  "image_name": "frame_00001.png"
}


## 4. Decide export mode

In [7]:
SOURCE_SIZE = (source['source_width'], source['source_height'])
USE_SYMLINKS = (TARGET_SIZE is None) or (TARGET_SIZE == SOURCE_SIZE)

if TARGET_SIZE is None:
    EXPORT_WIDTH, EXPORT_HEIGHT = SOURCE_SIZE
else:
    EXPORT_WIDTH, EXPORT_HEIGHT = TARGET_SIZE

print('SOURCE_SIZE =', SOURCE_SIZE)
print('EXPORT_SIZE =', (EXPORT_WIDTH, EXPORT_HEIGHT))
print('USE_SYMLINKS =', USE_SYMLINKS)

SOURCE_SIZE = (256, 256)
EXPORT_SIZE = (256, 256)
USE_SYMLINKS = True


## 5. Export DL3DV-2 in RegGS scene format

Key transformation: DL3DV-2 uses 1-indexed `frame_00001.png`, we export as 0-indexed `0000.png`.

In [8]:
def write_intrinsics_json(export_root: Path, intrinsics: dict):
    payload = {
        'fx': float(intrinsics['fx']),
        'fy': float(intrinsics['fy']),
        'cx': float(intrinsics['cx']),
        'cy': float(intrinsics['cy']),
    }
    (export_root / 'intrinsics.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')


def write_cameras_json(export_root: Path, cameras: list):
    """Write cameras.json with 0-indexed cam_id and image_name.
    
    DL3DV-2 source uses:
    - cam_id starting from 1
    - image_name like 'frame_00001.png'
    
    RegGS expects:
    - cam_id as 0-indexed (though it may not strictly require this)
    - image_name matching the exported filenames (0000.png, 0001.png, ...)
    """
    export_cameras = []
    for i, cam in enumerate(cameras):
        export_cameras.append({
            'cam_id': int(i),  # Convert to 0-indexed
            'cam_quat': [float(x) for x in cam['cam_quat']],
            'cam_trans': [float(x) for x in cam['cam_trans']],
            'cx': float(cam['cx']),
            'cy': float(cam['cy']),
            'fx': float(cam['fx']),
            'fy': float(cam['fy']),
            'image_name': f'{i:04d}.png',  # Convert to 0000.png format
        })
    (export_root / 'cameras.json').write_text(json.dumps(export_cameras, indent=2), encoding='utf-8')


def export_dl3dv_scene(source: dict, export_root: Path, overwrite: bool = True):
    if export_root.exists() and not overwrite:
        raise FileExistsError(f'{export_root} already exists and overwrite=False')
    safe_reset_dir(export_root)
    image_dir = export_root / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)

    if not USE_SYMLINKS:
        raise NotImplementedError('Resize/write mode is not implemented yet. Keep TARGET_SIZE=None for symlink export.')

    # Export images with 0-indexed naming
    for i, src in enumerate(source['frame_paths']):
        dst = image_dir / f'{i:04d}.png'
        symlink_file(src, dst)

    write_intrinsics_json(export_root, source['intrinsics'])
    write_cameras_json(export_root, source['cameras'])

    return {
        'export_root': export_root,
        'num_frames': len(source['frame_paths']),
        'use_symlinks': USE_SYMLINKS,
        'export_size': (EXPORT_WIDTH, EXPORT_HEIGHT),
    }


summary = export_dl3dv_scene(source, EXPORT_ROOT, overwrite=OVERWRITE_EXPORT)
summary

{'export_root': PosixPath('/home/bzhang512/CV_Project/dataset/DL3DV-2/part2/reggs_dl3dv2_fullseq_256'),
 'num_frames': 306,
 'use_symlinks': True,
 'export_size': (256, 256)}

## 6. Verify exported scene

In [9]:
export_cameras = json.loads((EXPORT_ROOT / 'cameras.json').read_text(encoding='utf-8'))
export_intrinsics = json.loads((EXPORT_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
export_images = sorted((EXPORT_ROOT / 'images').iterdir())

print('export root exists =', EXPORT_ROOT.exists())
print('num exported images =', len(export_images))
print('intrinsics =', export_intrinsics)
print('num camera entries =', len(export_cameras))
print('first exported camera =')
print(json.dumps(export_cameras[0], indent=2))
print('last exported camera =')
print(json.dumps(export_cameras[-1], indent=2))

print()
print('first five exported image entries:')
for p in export_images[:5]:
    target = p.resolve() if p.is_symlink() else p
    print(f' - {p.name} -> {target.name}')

export root exists = True
num exported images = 306
intrinsics = {'fx': 0.7970605911654538, 'fy': 0.7972243051546298, 'cx': 0.5, 'cy': 0.5}
num camera entries = 306
first exported camera =
{
  "cam_id": 0,
  "cam_quat": [
    0.021004781126976013,
    0.021099701523780823,
    0.01128463540226221,
    0.9994930028915405
  ],
  "cam_trans": [
    -4.861327171325684,
    -0.5403552055358887,
    -2.9286084175109863
  ],
  "cx": 0.5,
  "cy": 0.5,
  "fx": 0.7970605911654538,
  "fy": 0.7972243051546298,
  "image_name": "0000.png"
}
last exported camera =
{
  "cam_id": 305,
  "cam_quat": [
    0.00011230228119529784,
    0.11494671553373337,
    0.051628611981868744,
    0.992029070854187
  ],
  "cam_trans": [
    -6.40526008605957,
    -0.1172642856836319,
    -2.6146106719970703
  ],
  "cx": 0.5,
  "cy": 0.5,
  "fx": 0.7970605911654538,
  "fy": 0.7972243051546298,
  "image_name": "0305.png"
}

first five exported image entries:
 - 0000.png -> frame_00001.png
 - 0001.png -> frame_00002.png


## 7. Minimal RegGS compatibility check

In [10]:
assert (EXPORT_ROOT / 'images').exists()
assert (EXPORT_ROOT / 'intrinsics.json').exists()
assert (EXPORT_ROOT / 'cameras.json').exists()
assert len(export_images) == len(export_cameras) == len(source['frame_paths'])
assert export_images[0].name == '0000.png', f'First image should be 0000.png, got {export_images[0].name}'
assert export_cameras[0]['image_name'] == '0000.png', f'First camera image_name should be 0000.png, got {export_cameras[0]["image_name"]}'
assert export_cameras[0]['cam_id'] == 0, f'First cam_id should be 0, got {export_cameras[0]["cam_id"]}'
print('RegGS-ready scene check passed.')

RegGS-ready scene check passed.
